# 一般旅館營運月報 — 資料讀取與清洗

一般旅館月報以 ODS 格式提供，每年一個檔案，內含 12 個 sheet（每月一張）。
不同年份的欄位名稱有差異（中英混雜、空白、命名不一致）。

本 notebook 負責：
1. 逐年讀取 ODS 檔案，從每個 sheet 中定位表頭與資料範圍
2. 統一欄位名稱（透過 rename_map 對照表）
3. 合併為單一 DataFrame 並輸出為 `一般旅館_all.xlsx`

In [1]:
import pandas as pd
import numpy as np


In [2]:
def clean_df(year,df_dict):
    df_concat =pd.DataFrame()
    for month,df in  df_dict.items():
        mask = df.apply(lambda col: col.astype(str).str.contains('填報率', na=False)).any(axis=1)
        if mask.sum() == 0:
            print(f"⚠️ {year} 年 sheet {month} 找不到 '填報率'，跳過")
            continue
        header_idx = df[mask].index[0]

        mask = df.isin(["連江縣"]).any(axis=1)
        if mask.sum() == 0:
            print(f"⚠️ {year} 年 sheet {month} 找不到 '連江縣'，跳過")
            continue
        end_idx = df[mask].index[0]

        df.columns = df.iloc[header_idx]
        df = df.iloc[header_idx + 1:end_idx+1].reset_index(drop=True)
        df["年月"]= str(year)+ "_"+ str(month+1)

        df.columns = [col if '總家數' not in str(col) else '總家數' for col in df.columns]
    
        df_concat = pd.concat([df_concat, df], ignore_index=True)
    return df_concat

In [ ]:
year_dict= {}
for year in range(2017,2026):
        df_dict = pd.read_excel(f'一般旅館/{year}.ods', sheet_name=list(range(12)) ,engine='odf')  # ex:2017.ods, sheet_name依照月份區分
        year_dict[year]= clean_df(year,df_dict)

In [4]:
for year,df in year_dict.items():
    print(year , df.columns.to_list())
    print("-"*60)

2017 ['填報率', '縣市', '總出租客房數', '客房住用數', '客房住用率', '住宿人數', '平均房價', '客房收入', '餐飲收入', '其他收入', '收入合計', '裝修及設備支出', '員工人數', '未報家數', '年月']
------------------------------------------------------------
2018 ['填報率', '縣市', '總家數', '總出租客房數', '客房住用數', '客房住用率', '住宿人數', '平均房價', '客房收入', '餐飲收入', '其他收入', '收入合計', '裝修及設備支出', '員工人數', '未報家數', '年月']
------------------------------------------------------------
2019 ['填報率', '縣市', np.float64(nan), '總家數', '總出租客房數', '客房住用數', '客房住用率', '住宿人數', '平均房價', '客房收入', '餐飲收入', '其他收入', '收入合計', '裝修及設備支出', '員工人數', '未報家數', '年月']
------------------------------------------------------------
2020 ['填報率', '縣市', np.float64(nan), '總出租客房數', '客房住用數', '客房住用率', '住宿人數', '平均房價', '客房收入', '餐飲收入', '其他收入', '收入合計', '裝修及設備支出', '員工人數', '未報家數', '年月']
------------------------------------------------------------
2021 ['填報率', '縣市', '總出租客房數', '客房住用數', '客房住用率', '平均房價', '客房收入', '餐飲收入', '其他收入', '收入合計', '員工人數', '未報家數', '年月']
------------------------------------------------------------
2022 ['地區名稱', '填報率', '未報家數

In [ ]:
import numpy as np
# 統一欄位名稱的對照表（舊名 : 新名）
rename_map = {
    '縣市': '地區',
    '地區名稱': '地區',
    '地區名稱Region': '地區',
    '填報率%': '填報率',
    '客房住用數No. of Rooms Occupied': '客房住用數',
    '客房住用率': '住用率',
    '住用率Occupancy Rate': '住用率',
    '平均房價Average Room Rate': '平均房價',
    '房租收入Room Revenue': '客房收入',
    '房租收入': '客房收入',
    '餐飲收入F & B Revenue': '餐飲收入',
    '收入合計': '總營收',
    '總營業收入': '總營收',
    '總營業收入Total Revenue': '總營收',
}

# 要保留的 9 個欄位
keep_cols = ['地區', '填報率', '客房住用數', '住用率', '平均房價', '客房收入', '餐飲收入', '總營收', '年月']

cleaned_dict = {}

for year, df in year_dict.items():
    temp = df.copy()

    # 先去掉 nan 欄位（2019, 2020 的問題）
    temp = temp.loc[:, temp.columns.notna()]

    # 欄位名稱去空白
    temp.columns = temp.columns.str.strip()

    # 統一rename
    temp = temp.rename(columns=rename_map)

    # 只保留共同欄位
    temp = temp [keep_cols] 

    cleaned_dict[year] = temp

# 合併成一張總表
df_all = pd.concat(cleaned_dict.values(), ignore_index=True)
df_all

,地區,填報率,客房住用數,住用率,平均房價,客房收入,餐飲收入,總營收,年月
0,新北市,1.0,160067,0.4485,2143,343059573,132035432,587637621,2017_1
1,臺北市,1.0,473573,0.5919,3026,1433085607,376703290,1984884185,2017_1
2,桃園市,1.0,138053,0.4542,1791,247266673,123772098,421898567,2017_1
3,臺中市,1.0,270666,0.5025,2526,683751530,182220530,937357684,2017_1
4,臺南市,1.0,115303,0.4249,2605,300405815,139768686,472063380,2017_1
...,...,...,...,...,...,...,...,...,...
2371,基隆市,1.0,17231,0.4552,1966,33878259,4434053,41634640,2025_12
2372,新竹市,1.0,62315,0.5961,2122,132223470,75546592,220560006,2025_12
2373,嘉義市,1.0,90842,0.5888,2241,203536201,40119103,255848426,2025_12
2374,金門縣,1.0,13757,0.2958,2000,27514060,8046955,36829617,2025_12


In [ ]:
# 轉換年月為datetime
df_all['年月'] = pd.to_datetime(df_all['年月'], format='%Y_%m').dt.to_period('M')

# 統一欄位名稱
df_all.columns= ['地區', '填報率', '住用數', '住用率', '平均房價', '客房收入', '餐廳收入', '總營業收入', '年月']

In [ ]:
df_all.to_excel("一般旅館_all.xlsx",index=False)